# Laboratorio 13 — Aprendizaje por Refuerzo
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Noviembre de 2026, por definir · Recuperativa 1 (fuera del programa oficial)
**Bloques:** por definir

## Objetivos de aprendizaje

1. Distinguir el aprendizaje por refuerzo del supervisado y del no supervisado.
2. Formular un problema como proceso de decisión de Markov: estados, acciones, recompensas, política.
3. Explicar la función de valor, la ecuación de Bellman y el dilema exploración–explotación.
4. Implementar Q-learning tabular y entender por qué se pasa a redes (DQN) y a métodos de política.

## Plan de la sesión

| Parte | Contenido | Tiempo |
|---|---|---|
| A | Conocer el entorno FrozenLake y una política aleatoria | 15 min |
| B | Q-learning tabular en FrozenLake determinista | 35 min |
| C | Entorno estocástico y sensibilidad a hiperparámetros | 25 min |
| D | De la tabla a la red: DQN en CartPole *(opcional)* | 30 min |
| E | Reflexión | 5 min |

## Configuración

La variable `RAPIDO` reduce el número de episodios para probar el código en pocos minutos. Con `RAPIDO = False` se usan los valores del libreto.

In [ ]:
import sys, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "gymnasium>=1.0"

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42                # semilla global para reproducir los resultados
RAPIDO = False           # True: versión reducida para depurar el código rápido
EJECUTAR_D = True        # La Parte D (DQN) es opcional: pon False si no hay tiempo

# Tamaños de los experimentos (valores del libreto cuando RAPIDO = False)
N_EP_B = 3000 if not RAPIDO else 1000           # Parte B: episodios de Q-learning
N_EP_C = 20000 if not RAPIDO else 5000          # Parte C: episodios en entorno estocástico
N_EP_BARRIDO = 5000 if not RAPIDO else 2000     # Parte C: episodios por corrida del barrido
N_SEMILLAS = 5                                  # Parte C y E: semillas por configuración
N_EVAL = 10000 if not RAPIDO else 2000          # episodios de evaluación greedy
N_EP_DQN = 300 if not RAPIDO else 120           # Parte D: episodios de DQN

plt.rcParams.update({
    "figure.facecolor": "#f2f1ee", "axes.facecolor": "#fffefb", "axes.edgecolor": "#8b929c",
    "axes.labelcolor": "#1b2740", "axes.titlecolor": "#1b2740", "text.color": "#1b2740",
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True,
    "grid.color": "#dcdad3", "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 110, "font.size": 10,
    "axes.prop_cycle": plt.cycler(color=["#1b2740", "#c8807a", "#8b929c", "#9aa6b8"]),
})
print("gymnasium", gym.__version__, "| RAPIDO =", RAPIDO)

### Utilidades (ya escritas)

Estas funciones se entregan hechas: medir una política, dibujarla y calcular la tasa de éxito exacta. Léelas, pero no hace falta modificarlas.

- `exito_exacto` calcula, **sin muestreo**, la probabilidad de llegar a la meta en a lo más 100 pasos con la política *greedy* de una tabla `Q`, usando el modelo de transiciones `env.unwrapped.P` de FrozenLake (solo se puede hacer porque el entorno es pequeño y conocido; el agente nunca lo usa para aprender).
- `techo_dp` hace lo mismo con la mejor política posible (programación dinámica): es el techo teórico de éxito.

In [ ]:
ACCIONES = {0: "←", 1: "↓", 2: "→", 3: "↑"}   # FrozenLake: 0 izquierda, 1 abajo, 2 derecha, 3 arriba


def suavizar(x, ventana=100):
    """Media móvil simple de una serie 1D."""
    x = np.asarray(x, dtype=float)
    if len(x) < ventana:
        return x
    c = np.cumsum(np.insert(x, 0, 0.0))
    return (c[ventana:] - c[:-ventana]) / ventana


def epsilon_por_episodio(ep, n_episodes, eps_inicio=1.0, eps_fin=0.05, esquema="exp"):
    """Valor de epsilon en el episodio `ep`.

    esquema="exp": decae geométricamente de eps_inicio a eps_fin durante el 80 % de los episodios.
    esquema="lineal": decae linealmente en el mismo tramo.
    esquema=<número>: epsilon constante (p. ej. 0.1).
    """
    frac = min(ep / (0.8 * n_episodes), 1.0)
    if esquema == "exp":
        return eps_inicio * (eps_fin / eps_inicio) ** frac
    if esquema == "lineal":
        return eps_inicio - (eps_inicio - eps_fin) * frac
    return float(esquema)


def evaluar_greedy(env, Q, n_episodes=1000, seed=0):
    """Tasa de éxito (recompensa media) al jugar siempre la acción de mayor Q."""
    total = 0.0
    for i in range(n_episodes):
        s, _ = env.reset(seed=seed if i == 0 else None)
        fin = False
        while not fin:
            s, r, term, trunc, _ = env.step(int(Q[s].argmax()))
            fin = term or trunc
        total += r
    return total / n_episodes


def exito_exacto(env, Q, horizonte=100):
    """P(llegar a la meta en <= horizonte pasos) con la política greedy de Q (cálculo exacto)."""
    P, n_s = env.unwrapped.P, env.observation_space.n
    pi = Q.argmax(1)
    V = np.zeros(n_s)
    for _ in range(horizonte):
        V_nuevo = np.zeros(n_s)
        for s in range(n_s):
            for p, s2, r, term in P[s][pi[s]]:
                V_nuevo[s] += p * (r + (0.0 if term else V[s2]))
        V = V_nuevo
    return V[0]


def techo_dp(env, horizonte=100):
    """Mejor probabilidad de éxito posible en <= horizonte pasos (programación dinámica)."""
    P, n_s, n_a = env.unwrapped.P, env.observation_space.n, env.action_space.n
    V = np.zeros(n_s)
    for _ in range(horizonte):
        V_nuevo = np.zeros(n_s)
        for s in range(n_s):
            V_nuevo[s] = max(sum(p * (r + (0.0 if term else V[s2])) for p, s2, r, term in P[s][a])
                             for a in range(n_a))
        V = V_nuevo
    return V[0]


def dibujar_politica(Q, desc, ax=None, titulo="Política greedy"):
    """Dibuja la grilla con una flecha por casilla (acción de mayor Q). '·' = Q todavía en cero."""
    ax = ax or plt.gca()
    mapa = np.array([[c.decode() for c in fila] for fila in desc])
    n = mapa.shape[0]
    colores = {"S": "#9aa6b8", "F": "#fffefb", "H": "#1b2740", "G": "#c8807a"}
    for i in range(n):
        for j in range(n):
            ax.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, facecolor=colores[mapa[i, j]], edgecolor="#8b929c"))
            s = i * n + j
            if mapa[i, j] in "SF":
                txt = ACCIONES[int(Q[s].argmax())] if Q[s].any() else "·"
                ax.text(j + 0.5, n - 1 - i + 0.5, txt, ha="center", va="center", fontsize=20, color="#1b2740")
            else:
                ax.text(j + 0.5, n - 1 - i + 0.5, mapa[i, j], ha="center", va="center", fontsize=14, color="#fffefb")
    ax.set_xlim(0, n); ax.set_ylim(0, n); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(titulo)

## Parte A — Conocer el entorno (15 min)

`FrozenLake-v1` es una cuadrícula 4×4: **S** inicio, **F** hielo seguro, **H** hoyo (el episodio termina sin recompensa) y **G** meta (recompensa +1). Con `is_slippery=False` el movimiento es determinista.

1. Crea el entorno e imprime `observation_space` (16 estados) y `action_space` (4 acciones).
2. Juega 5 episodios con acciones al azar y observa cómo terminan.
3. Mide la tasa de éxito de la política aleatoria con muchos episodios (con 5 episodios no se puede estimar un porcentaje de 1–2 %).

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
print("observation_space:", env.observation_space)
print("action_space:     ", env.action_space)
print("Mapa (filas de arriba hacia abajo):")
for fila in env.unwrapped.desc:
    print("  ", "".join(c.decode() for c in fila))

In [ ]:
rng = np.random.default_rng(SEED)


def episodio_aleatorio(env, rng):
    """Juega un episodio eligiendo acciones al azar. Devuelve (recompensa_total, pasos)."""
    s, _ = env.reset()
    terminado = truncado = False
    total, pasos = 0.0, 0
    # TODO: mientras el episodio no termine (ni por terminación ni por truncamiento):
    #   - elige una acción al azar con rng.integers(...)
    #   - avanza con env.step(a) y acumula la recompensa y el número de pasos
    ...
    return total, pasos


for i in range(5):
    r, pasos = episodio_aleatorio(env, rng)
    print(f"Episodio {i + 1}: recompensa = {r:.0f}, pasos = {pasos}")

In [ ]:
n_aleatorios = 10000
exitos = np.mean([episodio_aleatorio(env, rng)[0] for _ in range(n_aleatorios)])
print(f"Tasa de éxito de la política aleatoria ({n_aleatorios} episodios): {exitos:.1%}")
assert 0.0 <= exitos <= 0.1, "Revisa episodio_aleatorio: la tasa de éxito aleatoria debería ser un porcentaje bajo (1-2 %)."

## Parte B — Q-learning tabular (35 min)

Implementa el ciclo completo de Q-learning sobre `FrozenLake-v1` con `is_slippery=False`:

- Tabla `Q = np.zeros((16, 4))`.
- Parámetros: `alpha = 0.8`, `gamma = 0.95`, ε decreciente de 1,0 a 0,05 (la función `epsilon_por_episodio` ya está escrita) y **3000 episodios** (`N_EP_B`).
- En cada paso: elegir la acción ε-greedy → `env.step` → actualización de Bellman → pasar al siguiente estado.

La actualización de Bellman es

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$$

y cuando el episodio **terminó** (meta u hoyo) no hay estado siguiente que valga: el término `max Q(s', ·)` se descarta.

Después: grafica la recompensa media móvil (ventana 100) por episodio, muestra la tabla `Q` final como mapa de calor, dibuja la política (flecha de mayor `Q` por casilla) y comprueba que llega a la meta.

In [ ]:
def actualizar_q(Q, s, a, r, s2, terminado, alpha, gamma):
    """Actualización de Bellman de Q-learning. Modifica Q in situ y devuelve el nuevo Q[s, a].

    Si `terminado` es True no hay estado siguiente: el término max Q(s', ·) no se suma.
    """
    # TODO 1: calcula el objetivo  r + gamma * max_a' Q[s2, a']  (sin el segundo término si terminado)
    objetivo = ...
    # TODO 2: mueve Q[s, a] una fracción alpha hacia el objetivo
    ...
    return float(Q[s, a])


def entrenar_q(env, n_episodes, alpha, gamma, eps_inicio=1.0, eps_fin=0.05, esquema="exp", seed=0):
    """Q-learning tabular. Devuelve (Q, recompensas_por_episodio)."""
    rng = np.random.default_rng(seed)
    n_estados, n_acciones = env.observation_space.n, env.action_space.n
    Q = np.zeros((n_estados, n_acciones))
    recompensas = np.zeros(n_episodes)
    for ep in range(n_episodes):
        eps = epsilon_por_episodio(ep, n_episodes, eps_inicio, eps_fin, esquema)
        s, _ = env.reset(seed=seed if ep == 0 else None)
        terminado = truncado = False
        while not (terminado or truncado):
            # TODO 3: política ε-greedy.
            #   Con probabilidad eps -> acción al azar (rng.integers); en otro caso -> argmax de Q[s]
            a = ...
            # TODO 4: avanza un paso en el entorno y desempaca (s2, r, terminado, truncado, info)
            ...
            # TODO 5: actualiza Q con actualizar_q(...), pasa al siguiente estado y acumula la recompensa
            ...
    return Q, recompensas

**Autocomprobación de la actualización de Bellman.** Con `Q(s,a) = 0`, `r = −1`, `γ = 0,9`, `max Q(s', ·) = 2` y `α = 0,5` el nuevo valor debe ser `0 + 0,5·(−1 + 1,8 − 0) = 0,4` (es el ejemplo de la diapositiva 9).

In [ ]:
Q_t = np.zeros((3, 2))
Q_t[1] = [2.0, 1.0]                                   # max Q(s', ·) = 2 para s' = 1
nuevo = actualizar_q(Q_t, 0, 0, -1.0, 1, False, 0.5, 0.9)
assert abs(nuevo - 0.4) < 1e-9 and abs(Q_t[0, 0] - 0.4) < 1e-9, f"Se esperaba 0.4 y se obtuvo {nuevo}"

Q_t2 = np.zeros((3, 2))
Q_t2[1] = [2.0, 1.0]
nuevo2 = actualizar_q(Q_t2, 0, 1, 1.0, 1, True, 0.5, 0.9)   # episodio terminado: no se suma el futuro
assert abs(nuevo2 - 0.5) < 1e-9, f"Con terminado=True se esperaba 0.5 y se obtuvo {nuevo2}"
print("OK: la actualización de Bellman da 0.4 (y 0.5 cuando el episodio terminó).")

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
t0 = time.time()
Q_b, rec_b = entrenar_q(env, N_EP_B, alpha=0.8, gamma=0.95, seed=SEED)
print(f"Entrenado en {time.time() - t0:.1f} s")
assert Q_b.shape == (16, 4), "La tabla Q debe tener forma (16, 4)"

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (1) recompensa media móvil
axes[0].plot(suavizar(rec_b, 100), color="#1b2740")
axes[0].set(title="Recompensa media móvil (ventana 100)", xlabel="episodio", ylabel="recompensa media")

# (2) mapa de calor de la tabla Q
im = axes[1].imshow(Q_b, cmap="Blues", aspect="auto")
axes[1].set(title="Tabla Q (estados × acciones)", xlabel="acción", ylabel="estado")
axes[1].set_xticks(range(4)); axes[1].set_xticklabels([ACCIONES[a] for a in range(4)]); axes[1].grid(False)
fig.colorbar(im, ax=axes[1], fraction=0.046)

# (3) política
dibujar_politica(Q_b, env.unwrapped.desc, ax=axes[2], titulo="Política greedy")
plt.tight_layout(); plt.show()

In [ ]:
exito_b = evaluar_greedy(env, Q_b, n_episodes=100)
print(f"Éxito de la política greedy (determinista, 100 episodios): {exito_b:.0%}")
if exito_b < 1.0:
    print("Ojo: la política no llega siempre a la meta. Q-learning es estocástico: si tu código es correcto, "
          "a veces basta con cambiar SEED o subir N_EP_B; si falla siempre, revisa entrenar_q y actualizar_q.")

## Parte C — Entorno estocástico y sensibilidad (25 min)

Con `is_slippery=True` el agente **resbala**: la acción elegida solo se cumple con probabilidad 1/3 y en las otras dos direcciones perpendiculares se desliza. Ahora el mismo valor de `Q` tiene que promediar sobre muchos resultados posibles.

**C.1** Entrena con **20 000 episodios** (`N_EP_C`) dos configuraciones: (a) las de la Parte B (`alpha = 0.8`, `gamma = 0.95`) y (b) `alpha = 0.1`, `gamma = 0.99`. Evalúa cada política de dos maneras:

- evaluación *greedy* con **N = 10 000 episodios** (`N_EVAL`; 2 000 si `RAPIDO = True`), y
- el cálculo exacto (`exito_exacto`), que no tiene ruido de muestreo.

Compáralas con el techo teórico `techo_dp`: ni la mejor política posible llega al 100 %.

In [ ]:
env_slip = gym.make("FrozenLake-v1", is_slippery=True)
t0 = time.time()
# TODO: entrena dos veces con entrenar_q sobre env_slip y N_EP_C episodios (usa seed=SEED):
#   (a) alpha=0.8, gamma=0.95  -> Q_c1, rec_c1
#   (b) alpha=0.1, gamma=0.99  -> Q_c2, rec_c2
Q_c1, rec_c1 = ...
Q_c2, rec_c2 = ...
print(f"Entrenado en {time.time() - t0:.1f} s")
assert Q_c1.shape == (16, 4) and Q_c2.shape == (16, 4)

In [ ]:
filas = []
for nombre, Q in [("(a) α=0.8, γ=0.95", Q_c1), ("(b) α=0.1, γ=0.99", Q_c2)]:
    filas.append({
        "configuración": nombre,
        f"éxito greedy ({N_EVAL} episodios)": evaluar_greedy(env_slip, Q, n_episodes=N_EVAL, seed=SEED),
        "éxito exacto": exito_exacto(env_slip, Q),
    })
filas.append({"configuración": "techo teórico (mejor política posible)",
              f"éxito greedy ({N_EVAL} episodios)": np.nan, "éxito exacto": techo_dp(env_slip)})
tabla_c1 = pd.DataFrame(filas).set_index("configuración")
display(tabla_c1.style.format("{:.1%}", na_rep="—"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(suavizar(rec_c1, 500), label="(a) α=0.8, γ=0.95")
axes[0].plot(suavizar(rec_c2, 500), label="(b) α=0.1, γ=0.99")
axes[0].set(title="Recompensa media móvil (ventana 500)", xlabel="episodio", ylabel="recompensa media")
axes[0].legend()
dibujar_politica(Q_c1, env_slip.unwrapped.desc, ax=axes[1], titulo="Política (a)")
dibujar_politica(Q_c2, env_slip.unwrapped.desc, ax=axes[2], titulo="Política (b)")
plt.tight_layout(); plt.show()

**Cuidado con comparar con una sola semilla.** La tabla anterior es de *una* corrida por configuración. Repite el entrenamiento con `N_SEMILLAS` semillas distintas y mira la media ± desviación estándar del éxito exacto (tarda ~1 min).

In [ ]:
t0 = time.time()
exactos = {"(a) α=0.8, γ=0.95": [], "(b) α=0.1, γ=0.99": []}
for sem in range(N_SEMILLAS):
    exactos["(a) α=0.8, γ=0.95"].append(exito_exacto(env_slip, entrenar_q(env_slip, N_EP_C, 0.8, 0.95, seed=sem)[0]))
    exactos["(b) α=0.1, γ=0.99"].append(exito_exacto(env_slip, entrenar_q(env_slip, N_EP_C, 0.1, 0.99, seed=sem)[0]))
print(f"{N_SEMILLAS} semillas × 2 configuraciones en {time.time() - t0:.0f} s")
for nombre, v in exactos.items():
    print(f"{nombre}: media = {np.mean(v):.1%} ± {np.std(v, ddof=1):.1%} | por semilla: {np.round(v, 2).tolist()}")

**Pregunta 1:** Aun con la mejor política posible, ¿por qué la tasa de éxito no llega a 100 %? ¿Qué muestra la comparación con 5 semillas sobre las configuraciones (a) y (b)? ¿Por qué la (a), que funcionaba en la Parte B, rinde peor aquí?

**Respuesta:** (escribe aquí)

**C.2 — Barrido de hiperparámetros.** Escribe la función `barrido` que, para cada combinación de `alpha ∈ {0.1, 0.5, 0.9}` y `gamma ∈ {0.5, 0.99}`, entrene `N_SEMILLAS = 5` semillas distintas con `N_EP_BARRIDO` episodios cada una y guarde el éxito exacto de la política resultante. Presenta la media ± desviación estándar en una tabla (herramienta de la clase 9: informar con incertidumbre).

**C.3 — Esquemas de ε.** Con la mejor combinación (`alpha = 0.1`, `gamma = 0.99`) compara tres esquemas de exploración: decaimiento exponencial (`"exp"`), lineal (`"lineal"`) y constante `0.1`.

In [ ]:
def barrido(env, alphas, gammas, esquemas, n_episodes, n_semillas):
    """Entrena una corrida por (alpha, gamma, esquema, semilla). Devuelve un DataFrame con el éxito exacto.

    Columnas del DataFrame: alpha, gamma, esquema (como texto), semilla, exito.
    """
    filas = []
    # TODO: recorre todas las combinaciones (itertools.product) y todas las semillas;
    #   entrena con entrenar_q(env, n_episodes, alpha, gamma, esquema=esquema, seed=semilla)
    #   y agrega a `filas` un diccionario con las columnas indicadas (éxito = exito_exacto(env, Q)).
    ...
    return pd.DataFrame(filas)


def resumen(df, filas, columnas):
    """Tabla 'media ± desv.' por celda."""
    g = df.groupby(filas + columnas)["exito"].agg(["mean", "std"]).reset_index()
    g["texto"] = g.apply(lambda f: f"{f['mean']:.1%} ± {f['std']:.1%}", axis=1)
    return g.pivot(index=filas, columns=columnas, values="texto")

In [ ]:
t0 = time.time()
df_ag = barrido(env_slip, alphas=[0.1, 0.5, 0.9], gammas=[0.5, 0.99], esquemas=["exp"],
                n_episodes=N_EP_BARRIDO, n_semillas=N_SEMILLAS)
print(f"Barrido α×γ: {len(df_ag)} corridas de {N_EP_BARRIDO} episodios en {time.time() - t0:.0f} s")
assert len(df_ag) == 3 * 2 * N_SEMILLAS, "Se esperaban 3 α × 2 γ × N_SEMILLAS corridas"
display(resumen(df_ag, ["alpha"], ["gamma"]))

In [ ]:
t0 = time.time()
df_eps = barrido(env_slip, alphas=[0.1], gammas=[0.99], esquemas=["exp", "lineal", 0.1],
                 n_episodes=N_EP_BARRIDO, n_semillas=N_SEMILLAS)
print(f"Barrido de esquemas ε: {time.time() - t0:.0f} s")
display(resumen(df_eps, ["esquema"], ["alpha"]))

**Pregunta 2:** Según el barrido, ¿qué combinación de `alpha` y `gamma` funciona mejor y por qué? ¿Qué le pasa a la variabilidad entre semillas cuando `alpha` es grande? ¿Cambian mucho los esquemas de ε?

**Respuesta:** (escribe aquí)

## Parte D — De la tabla a la red: DQN en CartPole (30 min, opcional)

En `CartPole-v1` el estado son 4 números **continuos** (posición y velocidad del carro, ángulo y velocidad angular del palo): no cabe en una tabla. La solución es aproximar `Q(s, a)` con una red neuronal. Hay dos trucos que lo hicieron funcionar (DeepMind, 2015):

- **Experience replay:** guardar las transiciones `(s, a, r, s', terminado)` en un buffer y entrenar con muestras al azar, para romper la correlación entre pasos consecutivos.
- **Red objetivo:** una copia congelada de la red que calcula el objetivo `r + γ max Q_objetivo(s', ·)` y se actualiza cada cierto número de pasos.

Parámetros del libreto: red `4 → 64 → 64 → 2` con ReLU, buffer de 10 000 transiciones, red objetivo actualizada cada 100 pasos, Adam con `lr = 1e-3`, ~300 episodios (`N_EP_DQN`). Se agregan: lote de 64, `gamma = 0.99`, ε decreciente de 1 a 0,05 en 5 000 pasos, pérdida Huber y recorte de gradiente en 10.

Primero la **línea base**: la política aleatoria obtiene una recompensa pequeña. El techo del entorno es 500.

In [ ]:
if EJECUTAR_D:
    import torch
    import torch.nn as nn
    torch.set_num_threads(1)
    torch.manual_seed(SEED)
    env_cp = gym.make("CartPole-v1")
    rng_cp = np.random.default_rng(SEED)

    def retorno_aleatorio(env, rng):
        s, _ = env.reset()
        total, fin = 0.0, False
        while not fin:
            s, r, term, trunc, _ = env.step(int(rng.integers(env.action_space.n)))
            total += r
            fin = term or trunc
        return total

    env_cp.reset(seed=SEED)                       # fija la semilla del entorno para que la línea base sea reproducible
    ret_aleatorio = np.array([retorno_aleatorio(env_cp, rng_cp) for _ in range(100)])
    print(f"Política aleatoria (100 episodios): recompensa media = {ret_aleatorio.mean():.1f}, máxima = {ret_aleatorio.max():.0f}")

**D.1 Piezas del DQN.** Completa la red, el buffer de repetición y el cálculo del objetivo. Las autocomprobaciones verifican formas y el valor del objetivo con un ejemplo a mano.

In [ ]:
if EJECUTAR_D:
    def crear_red(n_entradas=4, n_acciones=2):
        """Red Q: n_entradas -> 64 -> 64 -> n_acciones con ReLU."""
        # TODO: devuelve un nn.Sequential con dos capas ocultas de 64 neuronas y activación ReLU
        ...

    class BufferRepeticion:
        """Buffer circular de transiciones (s, a, r, s', terminado)."""
        def __init__(self, capacidad, dim_estado):
            self.capacidad, self.n, self.ptr = capacidad, 0, 0
            self.s = np.zeros((capacidad, dim_estado), np.float32)
            self.a = np.zeros(capacidad, np.int64)
            self.r = np.zeros(capacidad, np.float32)
            self.s2 = np.zeros((capacidad, dim_estado), np.float32)
            self.t = np.zeros(capacidad, np.float32)

        def agregar(self, s, a, r, s2, terminado):
            # TODO: guarda la transición en la posición self.ptr, avanza ptr de forma circular
            #       (al llegar a `capacidad` vuelve a 0) y actualiza self.n (como máximo `capacidad`).
            ...

        def muestrear(self, lote, rng):
            # TODO: elige `lote` índices al azar entre 0 y self.n y devuelve la tupla de tensores
            #       (s, a, r, s2, t) con torch.from_numpy(...)
            ...

        def __len__(self):
            return self.n

    def calcular_objetivo(red_objetivo, r, s2, terminado, gamma):
        """y = r + gamma * max_a' Q_objetivo(s', a') * (1 - terminado). Sin gradiente."""
        # TODO: calcula el objetivo del DQN con la red objetivo (usa torch.no_grad())
        ...

In [ ]:
if EJECUTAR_D:
    # Autocomprobaciones
    red_t = crear_red()
    n_params = sum(p.numel() for p in red_t.parameters())
    assert n_params == 4 * 64 + 64 + 64 * 64 + 64 + 64 * 2 + 2, f"Parámetros de la red: {n_params}"
    assert red_t(torch.zeros(5, 4)).shape == (5, 2)

    buf_t = BufferRepeticion(3, 4)
    for k in range(5):                                        # 5 inserciones en un buffer de 3
        buf_t.agregar(np.full(4, k), k % 2, float(k), np.full(4, k + 1), k == 4)
    assert len(buf_t) == 3, "El buffer no debe superar su capacidad"
    s_, a_, r_, s2_, t_ = buf_t.muestrear(8, np.random.default_rng(0))
    assert s_.shape == (8, 4) and a_.shape == (8,) and t_.dtype == torch.float32
    assert set(r_.tolist()) <= {2.0, 3.0, 4.0}, "Con capacidad 3 deben quedar las 3 transiciones más recientes (k = 2, 3, 4)"

    # Objetivo con una red objetivo falsa: Q(s') = [[1, 3], [2, 0.5]]
    falsa = lambda x: torch.tensor([[1.0, 3.0], [2.0, 0.5]])
    y = calcular_objetivo(falsa, torch.tensor([1.0, 1.0]), torch.zeros(2, 4), torch.tensor([0.0, 1.0]), 0.9)
    assert torch.allclose(y, torch.tensor([3.7, 1.0])), f"Objetivo esperado [3.7, 1.0]; obtenido {y.tolist()}"
    print(f"OK: red con {n_params} parámetros, buffer circular y objetivo = {y.tolist()}")

**D.2 Entrenamiento.** Completa el paso de optimización dentro de `entrenar_dqn` (la selección ε-greedy y el almacenamiento en el buffer ya están escritos). Se guarda además la mejor red según la media móvil de los últimos 20 episodios, porque en DQN la política final puede ser peor que una intermedia.

In [ ]:
if EJECUTAR_D:
    def entrenar_dqn(env, n_episodes, gamma=0.99, lr=1e-3, lote=64, capacidad=10_000,
                     cada_target=100, paso_inicio=500, seed=SEED):
        """DQN con replay y red objetivo. Devuelve (retornos, mejor_red, red_final)."""
        rng = np.random.default_rng(seed)
        torch.manual_seed(seed)
        red, red_obj = crear_red(), crear_red()
        red_obj.load_state_dict(red.state_dict())
        opt = torch.optim.Adam(red.parameters(), lr=lr)
        buf = BufferRepeticion(capacidad, env.observation_space.shape[0])
        retornos, paso = [], 0
        mejor_media, mejor_red = -np.inf, None
        for ep in range(n_episodes):
            s, _ = env.reset(seed=seed if ep == 0 else None)
            terminado = truncado = False
            total = 0.0
            while not (terminado or truncado):
                eps = max(0.05, 1.0 - paso / 5000)                       # ε baja de 1 a 0.05 en 5000 pasos
                if rng.random() < eps:
                    a = int(rng.integers(env.action_space.n))
                else:
                    with torch.no_grad():
                        a = int(red(torch.as_tensor(s, dtype=torch.float32)).argmax())
                s2, r, terminado, truncado, _ = env.step(a)
                buf.agregar(s, a, r, s2, terminado)                       # truncado NO es estado terminal
                s, total, paso = s2, total + r, paso + 1
                if len(buf) >= paso_inicio:
                    S, A, R, S2, T = buf.muestrear(lote, rng)
                    # TODO 1: objetivo y con la red objetivo (calcular_objetivo)
                    y = ...
                    # TODO 2: Q(s, a) de las acciones tomadas: red(S) tiene forma (lote, 2); usa gather con A
                    q_sa = ...
                    # TODO 3: pérdida Huber (nn.functional.smooth_l1_loss) entre q_sa e y
                    perdida = ...
                    # TODO 4: un paso del optimizador (zero_grad, backward, recorte de gradiente a norma 10, step)
                    ...
                if paso % cada_target == 0:
                    red_obj.load_state_dict(red.state_dict())            # sincroniza la red objetivo
            retornos.append(total)
            if len(retornos) >= 20 and np.mean(retornos[-20:]) > mejor_media:
                mejor_media = np.mean(retornos[-20:])
                mejor_red = {k: v.clone() for k, v in red.state_dict().items()}
        return np.array(retornos), mejor_red, red

In [ ]:
if EJECUTAR_D:
    t0 = time.time()
    retornos, pesos_mejor, red_final = entrenar_dqn(env_cp, N_EP_DQN)
    print(f"DQN: {N_EP_DQN} episodios en {time.time() - t0:.0f} s")
    print(f"Recompensa media de los últimos 20 episodios: {retornos[-20:].mean():.1f} | máxima: {retornos.max():.0f}")

In [ ]:
if EJECUTAR_D:
    def evaluar_red(env, red, n=20, seed=123):
        """Recompensa media jugando siempre la acción de mayor Q (sin exploración)."""
        totales = []
        for i in range(n):
            s, _ = env.reset(seed=seed if i == 0 else None)
            total, fin = 0.0, False
            while not fin:
                with torch.no_grad():
                    a = int(red(torch.as_tensor(s, dtype=torch.float32)).argmax())
                s, r, term, trunc, _ = env.step(a)
                total += r
                fin = term or trunc
            totales.append(total)
        return float(np.mean(totales))

    red_mejor = crear_red(); red_mejor.load_state_dict(pesos_mejor)
    ev_mejor, ev_final = evaluar_red(env_cp, red_mejor), evaluar_red(env_cp, red_final)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(retornos, color="#9aa6b8", lw=0.8, label="recompensa por episodio")
    ax.plot(np.arange(19, len(retornos)), suavizar(retornos, 20), color="#1b2740", lw=2, label="media móvil (20)")
    ax.axhline(ret_aleatorio.mean(), color="#c8807a", ls="--", label=f"política aleatoria ({ret_aleatorio.mean():.0f})")
    ax.axhline(500, color="#8b929c", ls=":", label="techo del entorno (500)")
    ax.set(title="DQN en CartPole-v1", xlabel="episodio", ylabel="recompensa"); ax.legend(loc="upper left")
    plt.tight_layout(); plt.show()
    print(f"Evaluación greedy (20 episodios): mejor red = {ev_mejor:.1f} | red final = {ev_final:.1f} | aleatoria = {ret_aleatorio.mean():.1f}")

## Parte E — Reflexión (5 min)

Responde las tres preguntas. Las dos celdas siguientes son **experimentos de apoyo** (ya escritos) para fundamentar las respuestas 1 y 2; las puedes ejecutar y mirar antes de escribir.

In [ ]:
# Experimento de apoyo para la pregunta 1: ¿qué rol cumple ε?  (FrozenLake determinista, 5 semillas)
env_det = gym.make("FrozenLake-v1", is_slippery=False)
esquemas_e = {"ε decreciente (1 → 0.05)": "exp", "ε = 0 (nunca explora)": 0.0,
              "ε = 1 (siempre al azar)": 1.0, "ε = 0.1 constante": 0.1}
resultados_e = {}
for nombre, esq in esquemas_e.items():
    exitos_e = [exito_exacto(env_det, entrenar_q(env_det, N_EP_B, 0.8, 0.95, esquema=esq, seed=sem)[0])
                for sem in range(N_SEMILLAS)]
    resultados_e[nombre] = f"{np.mean(exitos_e):.0%} ± {np.std(exitos_e, ddof=1):.0%}"
display(pd.Series(resultados_e, name=f"éxito de la política greedy final ({N_SEMILLAS} semillas)").to_frame())

In [ ]:
# Experimento de apoyo para la pregunta 2: recompensa +1 por cada paso en que el agente no cae en un hoyo
class PasoSinHoyo(gym.Wrapper):
    """Recompensa alternativa: +1 en cada paso, salvo el paso en que se cae a un hoyo (0)."""
    def step(self, a):
        s, r, term, trunc, info = self.env.step(a)
        cayo = term and self.env.unwrapped.desc.flatten()[s] == b"H"
        return s, 0.0 if cayo else 1.0, term, trunc, info


env_hack = PasoSinHoyo(gym.make("FrozenLake-v1", is_slippery=False))
Q_h, _ = entrenar_q(env_hack, N_EP_B, alpha=0.8, gamma=0.95, seed=SEED)


def jugar_greedy(env, Q):
    s, _ = env.reset(seed=0)
    ret, pasos, fin = 0.0, 0, False
    while not fin:
        s, r, term, trunc, _ = env.step(int(Q[s].argmax()))
        ret, pasos, fin = ret + r, pasos + 1, term or trunc
    return ret, pasos, bool(term and env.unwrapped.desc.flatten()[s] == b"G")


ret_h, pasos_h, meta_h = jugar_greedy(env_hack, Q_h)
print(f"Con la recompensa +1 por paso, el agente greedy: retorno = {ret_h:.0f}, pasos = {pasos_h}, ¿llegó a la meta? {meta_h}")
dibujar_politica(Q_h, env_hack.unwrapped.desc, titulo="Política aprendida con la recompensa +1 por paso")
plt.show()

**Pregunta 1:** ¿Qué rol cumple ε? ¿Qué ocurre si es 0, si es 1 o si decrece?

**Respuesta:** (escribe aquí)

**Pregunta 2:** ¿Qué pasaría si la recompensa fuera +1 por cada paso sin caer en un hoyo? (mira el experimento de apoyo)

**Respuesta:** (escribe aquí)

**Pregunta 3:** ¿Qué cambia al pasar de una tabla a una red (DQN)? Apóyate en lo observado en CartPole (Parte D).

**Respuesta:** (escribe aquí)

## Entregable

Notebook con las **Partes A–C ejecutadas** (la D es opcional) y las **tres respuestas de la Parte E**. Entrega por AULA-USM.